# Bank Customer Churn: Cleaning
Raw data: `data/Customer-Churn-Records.csv` (Kaggle, 10,000 customers)

Output: `data/churn_clean.csv`

In [1]:
import pandas as pd
df = pd.read_csv("data/Customer-Churn-Records.csv")
print(df.shape)
print("missing values:", df.isnull().sum().sum())
print("duplicate rows:", df.duplicated().sum())
print("duplicate CustomerIds:", df["CustomerId"].duplicated().sum())
df.describe().T

(10000, 18)
missing values: 0
duplicate rows: 0
duplicate CustomerIds: 0


,count,mean,std,min,25%,50%,75%,max
RowNumber,10000.0,5.000500e+03,2886.895680,1.00,2500.75,5.000500e+03,7.500250e+03,10000.00
CustomerId,10000.0,1.569094e+07,71936.186123,15565701.00,15628528.25,1.569074e+07,1.575323e+07,15815690.00
CreditScore,10000.0,6.505288e+02,96.653299,350.00,584.00,6.520000e+02,7.180000e+02,850.00
Age,10000.0,3.892180e+01,10.487806,18.00,32.00,3.700000e+01,4.400000e+01,92.00
Tenure,10000.0,5.012800e+00,2.892174,0.00,3.00,5.000000e+00,7.000000e+00,10.00
Balance,10000.0,7.648589e+04,62397.405202,0.00,0.00,9.719854e+04,1.276442e+05,250898.09
NumOfProducts,10000.0,1.530200e+00,0.581654,1.00,1.00,1.000000e+00,2.000000e+00,4.00
HasCrCard,10000.0,7.055000e-01,0.455840,0.00,0.00,1.000000e+00,1.000000e+00,1.00
IsActiveMember,10000.0,5.151000e-01,0.499797,0.00,0.00,1.000000e+00,1.000000e+00,1.00
EstimatedSalary,10000.0,1.000902e+05,57510.492818,11.58,51002.11,1.001939e+05,1.493882e+05,199992.48


## Findings
- 10,000 rows, 18 columns, no missing values, no duplicate rows or CustomerIds.
- Ranges look sane (Age 18-92, CreditScore 350-850, Tenure 0-10).
- 3,617 customers (36%) have a zero balance, which matters for the balance tiers.
- `Complain` almost perfectly tracks `Exited` (99.5% of complainers left vs 0.05% of others), so it is leakage. Do not treat it as a churn driver or use it as a model feature.

In [2]:
df = df.drop(columns=["RowNumber", "CustomerId", "Surname"])
df = df.drop_duplicates()
df["AgeGroup"] = pd.cut(df["Age"], [17,30,45,60,100], labels=["18-30","31-45","46-60","60+"])
# rank first so the 3,617 zero balances split evenly across tiers instead of failing on duplicate edges
df["BalanceTier"] = pd.qcut(df["Balance"].rank(method="first"), 4, labels=["Low","Mid","High","Top"])
df.to_csv("data/churn_clean.csv", index=False)
df.shape

(10000, 17)